# Sensitivity Analysis

Analisi di sensitività degli esponenti di scaling ζ(q) rispetto a perturbazioni dei pick P/S.

## 1. Imports and setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from pathlib import Path
import pickle

from src import (
    set_plot_style,
    analyze_all_windows,
    segment_all_signals,
    run_sensitivity_analysis,
    create_summary
)

colors, colors1 = set_plot_style()
logging.basicConfig(level=logging.WARNING)

print("Environment ready")

Environment ready


## 2. Configuration

In [2]:
# Dataset
DATASET = 'current'
PICKING_METHOD = 'phasenet'

# ============================================================================
# CHOOSE DATA TYPE (change this and re-run notebook for each type)
# ============================================================================
DATA_TYPE = 'acceleration'  # OPTIONS: 'acceleration', 'velocity', 'displacement'
# ============================================================================

# Coda methods to analyze
CODA_METHODS = ['rautian', 'arias', 'envelope', 'median']

# Perturbation scenarios
PERTURBATION_SCENARIOS = {
    'noise_small': {'type': 'gaussian', 'std': 0.2},
    'noise_medium': {'type': 'gaussian', 'std': 0.5},
    'noise_large': {'type': 'gaussian', 'std': 1.0},
    'bias_early': {'type': 'bias', 'bias': -0.5},
    'bias_late': {'type': 'bias', 'bias': 0.5}
}

# Monte Carlo parameters
N_MONTE_CARLO = 100
MONTE_CARLO_STD = 0.5

# Analysis parameters
TAU_MIN = 0.01
Q_VALUES = np.linspace(0.25, 5.0, 20)
SAMPLING_RATE = 200.0

# Signal columns
SIGNAL_COLUMN_MAP = {
    'acceleration': 'signal',
    'velocity': 'signal',
    'displacement': 'signal'
}

SIGNAL_COLUMN = SIGNAL_COLUMN_MAP[DATA_TYPE]

print(f"Configuration set for: {DATA_TYPE}")
print(f"Coda methods: {CODA_METHODS}")
print(f"Scenarios: {len(PERTURBATION_SCENARIOS)}")
print(f"Monte Carlo runs: {N_MONTE_CARLO}")

Configuration set for: acceleration
Coda methods: ['rautian', 'arias', 'envelope', 'median']
Scenarios: 5
Monte Carlo runs: 100


## 3. Load data

In [3]:
# Paths
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / 'data' / 'processed'
PICKS_DIR = DATA_DIR / f'03b_phase_identification_{PICKING_METHOD}' / DATA_TYPE
BASELINE_DIR = DATA_DIR / '04a_moment_scaling_spatial' / PICKING_METHOD / DATA_TYPE
OUTPUT_DIR = DATA_DIR / '05_sensitivity_analysis' / PICKING_METHOD / DATA_TYPE

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load signals
print(f"Loading data for: {DATA_TYPE}")
with open(PICKS_DIR / f'signals_{DATA_TYPE}_dict.pkl', 'rb') as f:
    signals_dict = pickle.load(f)
print(f"  Signals: {len(signals_dict)} stations")

# Load picks
df_full = pd.read_parquet(PICKS_DIR / f'df_full_{DATA_TYPE}_{PICKING_METHOD}.parquet')
print(f"  Picks: {len(df_full)} records")

# Load baseline results
baseline_results = {}
for coda_method in CODA_METHODS:
    baseline_file = BASELINE_DIR / coda_method / 'ensemble_spatial_summary.parquet'
    if baseline_file.exists():
        baseline_results[coda_method] = pd.read_parquet(baseline_file)
        print(f"  Baseline {coda_method}: loaded")

print("\nData loaded")

Loading data for: acceleration
  Signals: 21 stations
  Picks: 63 records
  Baseline rautian: loaded
  Baseline arias: loaded
  Baseline envelope: loaded
  Baseline median: loaded

Data loaded


## 4. Run sensitivity analysis

In [4]:
# Pack configuration
config = {
    'TAU_MIN': TAU_MIN,
    'Q_VALUES': Q_VALUES,
    'SAMPLING_RATE': SAMPLING_RATE,
    'SIGNAL_COLUMN': SIGNAL_COLUMN,
    'N_MONTE_CARLO': N_MONTE_CARLO,
    'MONTE_CARLO_STD': MONTE_CARLO_STD
}

# Run analysis
results = run_sensitivity_analysis(
    data_type=DATA_TYPE,
    signals_dict=signals_dict,
    df_full=df_full,
    baseline_results=baseline_results,
    coda_methods=CODA_METHODS,
    perturbation_scenarios=PERTURBATION_SCENARIOS,
    segment_function=segment_all_signals,
    analyze_function=analyze_all_windows,
    config=config,
    output_dir=OUTPUT_DIR,
    verbose=False
)


SENSITIVITY ANALYSIS: ACCELERATION
Coda methods: 4
Scenarios: 5 + Monte Carlo (100 runs)
Total combinations: 24


[ACCELERATION] Processing: rautian
--------------------------------------------------------------------------------
Saved: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/data/processed/05_sensitivity_analysis/phasenet/acceleration/rautian/noise_small/ensemble_spatial_moments_pre_event.parquet
Saved: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/data/processed/05_sensitivity_analysis/phasenet/acceleration/rautian/noise_small/ensemble_spatial_moments_p_wave.parquet
Saved: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/data/processed/05_sensitivity_analysis/phasenet/acceleration/rautian/noise_small/ensemble_spatial_moments_s_wave.parquet
Saved: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/data/processed/05_sensitivity_analysis/phasenet/acceleration/rautian/noise_small/ensemble_spatial_moments_coda.

## 5. Save results

In [5]:
# Save complete results
results_file = OUTPUT_DIR / f'sensitivity_results_{DATA_TYPE}.pkl'
with open(results_file, 'wb') as f:
    pickle.dump(results, f)
print(f"Saved complete results: {results_file}")

# Create and save summary
df_summary = create_summary(
    results,
    data_type=DATA_TYPE,
    save_path=OUTPUT_DIR / f'sensitivity_summary_{DATA_TYPE}.csv'
)

print(f"Summary shape: {df_summary.shape}")
print(f"Saved summary: {OUTPUT_DIR / f'sensitivity_summary_{DATA_TYPE}.csv'}")

Saved complete results: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/data/processed/05_sensitivity_analysis/phasenet/acceleration/sensitivity_results_acceleration.pkl
Summary shape: (80, 9)
Saved summary: /Users/giulianaparadiso/Desktop/PoliTo/Tesi/tesi-seismic-analysis/data/processed/05_sensitivity_analysis/phasenet/acceleration/sensitivity_summary_acceleration.csv


## 6. View summary

In [6]:
# Display summary pivot table
print(f"\nSUMMARY: {DATA_TYPE.upper()}")
print("="*80)

summary_pivot = df_summary.pivot_table(
    index=['scenario', 'window'],
    columns='coda_method',
    values='rmse',
    aggfunc='mean'
)

print(summary_pivot)

# Top 10 worst cases
print(f"\n\nTOP 10 WORST CASES (highest RMSE)")
print("="*80)

worst_cases = df_summary.nlargest(10, 'rmse')[[
    'coda_method', 'scenario', 'window', 'rmse', 'mae', 'correlation'
]]
print(worst_cases.to_string(index=False))


SUMMARY: ACCELERATION
coda_method                arias  envelope    median   rautian
scenario     window                                           
bias_early   coda       0.000000  0.000000  0.000000  0.000000
             p_wave     0.021416  0.021416  0.021416  0.021416
             pre_event  0.100737  0.100737  0.100737  0.100737
             s_wave     0.081740  0.081501  0.118740  0.093778
bias_late    coda       0.000000  0.000000  0.000000  0.000000
             p_wave     0.123710  0.123710  0.123710  0.123710
             pre_event  0.186863  0.186863  0.186863  0.186863
             s_wave     0.234562  0.262522  0.369440  0.332143
noise_large  coda       0.000000  0.000000  0.000000  0.000000
             p_wave     0.264936  0.264936  0.264936  0.264936
             pre_event  0.053934  0.053934  0.053934  0.053934
             s_wave     0.342809  0.222164  0.036319  0.061785
noise_medium coda       0.000000  0.000000  0.000000  0.000000
             p_wave     0.017187

## 7. Visualization